# SuttaPlayer Piper1 VITS Control Panel (v12)
This notebook implements the decoupled, single-responsibility **Source of Truth** architecture for running your T4 GPU training runs. It is completely optimized for the Colab native terminal workspace with automated pre-flight gating and zero-trash telemetry.

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive
!git clone https://github.com/dhamma-initiative/sutta-tts-model-training.git
!git switch "colab-trials"
%mkdir -p piper_training/checkpoints
%cd /content

In [ ]:
%%writefile /content/drive/MyDrive/piper_training/training-stage-manifest.csv
resource,destination,extract_to,id
en-gb_pisi-suttaplayer-medium-dataset.tgz,/content/drive/MyDrive/piper_training,/content/drive/MyDrive/piper_training,1pKHDI95OcCXvv4ZHR3YS5Rmf6yzMkc7K
piper_cache.tgz,/content/drive/MyDrive/piper_training,/content/piper_cache,1LpOwLRVmjbgfcuuzWAGUhndK610UN3f8
sutta_wheels_backup.tar.gz,/content/drive/MyDrive/piper_training,/content/wheels_local,1ANL7SEQQaSDGlM9S0evJM2FgM97z1tyE
piper1_compiled_backup.tar.gz,/content/drive/MyDrive/piper_training,/content/,11h_W4qCX9gYcwgY-Gx0F8CI_DlsxuOUK
,,,
last-9288-57498.ckpt,/content/drive/MyDrive/piper_training,,1gAM5lrHQf4crwcORTsj6RHFoRw6Fw4LD

### Step 2: Resource Acquisition & Environment Restore
Execute these two cells back-to-back. The first cell pulls large datasets, dependencies, and model states from shared GDrive IDs using shared manifests. The second cell performs a **100% offline pip install** from your local cache and links precompiled Cython/C++ folders locally in under 30 seconds.

In [ ]:
# Install Deno globally (takes ~2 seconds)
!curl -fsSL https://deno.land/install.sh | sh
import os
os.environ['PATH'] += ':/root/.deno/bin'

# Cell 2.1: Pull and Stage cloud assets from shared IDs
!deno run --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-gdown-resources.ts \
  -i /content/drive/MyDrive/piper_training/training-stage-manifest.csv

In [ ]:
# Cell 2.2: Rebuild PyTorch and link pre-compiled Cython/C++ libraries locally
!deno run --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-bootstrap.ts

### Step 3: Pre-Flight Cross-Check ("Green for Go" Gating Engine)
To prevent duplicate efforts or unrecoverable crashes due to sudden GDrive FUSE mount drops, the `PreFlightValidator` acts as a strict proxy. It parses your actual `train_cmd` string, extracts target paths, runs physical sanity assertions, checks your checkpoint metadata (Epoch/Global Step), and **only unlocks the training loop if all checks are 100% green**.

In [ ]:
import os
import shlex
import subprocess
import json
import torch
import pandas as pd

class PreFlightValidator:
    def __init__(self, command_str):
        self.command_str = command_str.strip()
        self.env_vars = os.environ.copy()
        self.args = []
        self.params = {}
        self._parse_command()

    def _parse_command(self):
        tokens = shlex.split(self.command_str)
        cmd_start_idx = 0
        for i, token in enumerate(tokens):
            if "=" in token and not token.startswith("-"):
                key, val = token.split("=", 1)
                self.env_vars[key] = val
                cmd_start_idx = i + 1
            else:
                break
        self.args = tokens[cmd_start_idx:]
        for i in range(len(self.args) - 1):
            flag = self.args[i]
            if flag.startswith("-"):
                self.params[flag] = self.args[i + 1]

    def run_audit(self):
        print("=" * 65)
        print("          SUTTAPLAYER LIVE 'GREEN FOR GO' PIPELINE AUDIT         ")
        print("=" * 65)
        
        csv_path = self.params.get("--data.csv_path")
        audio_dir = self.params.get("--data.audio_dir")
        map_path = self.params.get("--data.phonemes_path")
        ckpt_path = self.params.get("--ckpt_path")
        failures = []

        # 1. Phonetic Metadata CSV
        print("📋 1. Phonetic Metadata CSV:")
        if csv_path and os.path.exists(csv_path):
            try:
                with open(csv_path, "r", encoding="utf-8") as f:
                    sample_lines = [f.readline().strip() for _ in range(3)]
                row_count = sum(1 for _ in open(csv_path, "r", encoding="utf-8"))
                print(f"  [PASS] CSV parsed successfully at: {csv_path}")
                print(f"  [INFO] Total Dataset Utterances: {row_count}")
                print("  [INFO] Sample rows (visual check):")
                for line in sample_lines:
                     print(f"    👉 {line[:100]}..." if len(line) > 100 else f"    👉 {line")
            except Exception as e:
                failures.append(f"Metadata CSV read error: {e}")
        else:
            failures.append(f"Metadata CSV not found or missing from args. Path: {csv_path}")
        print("-" * 65)

        # 2. Raw Audio WAV Payload
        print("🔊 2. Raw Audio WAV Directory:")
        if audio_dir and os.path.exists(audio_dir):
            try:
                wavs = [f for f in os.listdir(audio_dir) if f.endswith(".wav")]
                wav_count = len(wavs)
                anchors = ["0.wav", "500.wav", "999.wav"]
                missing_anchors = [a for a in anchors if not os.path.exists(os.path.join(audio_dir, a))]
                if wav_count == 1000 and not missing_anchors:
                    print(f"  [PASS] Found exactly {wav_count} audio files in: {audio_dir}")
                    print("  [PASS] Anchor verification successful (0.wav, 500.wav, 999.wav are online).")
                else:
                    failures.append(f"Audio payload error: Found {wav_count}/1000 WAVs. Missing anchors: {missing_anchors}")
            except Exception as e:
                failures.append(f"Audio folder list error: {e}")
        else:
            failures.append(f"Audio directory not found or missing from args. Path: {audio_dir}")
        print("-" * 65)

        # 3. Phoneme Map JSON Keys
        print("🗺️ 3. Phoneme Map JSON Keys:")
        if map_path and os.path.exists(map_path):
            try:
                with open(map_path, "r", encoding="utf-8") as f:
                    pmap = json.load(f)
                map_keys = list(pmap.keys())
                print(f"  [PASS] Phoneme map loaded with {len(map_keys)} valid tokens.")
                print(f"  [INFO] Mapped token sample: {map_keys[:10]}")
            except Exception as e:
                failures.append(f"Phoneme map parse error: {e}")
        else:
            failures.append(f"Phoneme map JSON not found or missing from args. Path: {map_path}")
        print("-" * 65)

        # 4. Checkpoint Resume Verification
        print("💾 4. Target Resuming Checkpoint State:")
        if ckpt_path and os.path.exists(ckpt_path):
            try:
                checkpoint = torch.load(ckpt_path, map_location="cpu")
                if isinstance(checkpoint, dict):
                    epoch = checkpoint.get("epoch", "Unknown")
                    global_step = checkpoint.get("global_step", "Unknown")
                    print(f"  [PASS] Verified Checkpoint: {os.path.basename(ckpt_path)}")
                    print(f"  [INFO] Next training run will resume from: Epoch {epoch} | Step {global_step}")
                else:
                    failures.append("Checkpoint loaded but contains unexpected dict format.")
            except Exception as e:
                failures.append(f"Checkpoint state corruption: {e}")
        else:
            failures.append(f"Resume checkpoint not found or missing from args. Path: {ckpt_path}")
        print("=" * 65)

        if failures:
            print("\n❌ [CRITICAL FAILURES ENCOUNTERED] - Training blocked!")
            for fail in failures:
                print(f"  🚨 {fail}")
            print("=" * 65)
            raise AssertionError("Pre-Flight validation checks failed! Resolve errors above to proceed.")
        
        print("\n🟢 [GREEN FOR GO] All pre-flight parameters verified. Ready for launch!")
        print("=" * 65)

    def train(self):
        print("\n🚀 Spawning PyTorch Lightning training process inside the virtual environment...")
        result = subprocess.run(
            self.args,
            env=self.env_vars,
            stdout=None, # Inherits standard streams for real-time progress bars
            stderr=None
        )
        if result.returncode != 0:
            raise RuntimeError(f"Trainer process terminated with non-zero exit code: {result.returncode}")

### Step 4: Define training command & execute with Pre-Flight Check
If you are running in foreground mode inside your browser cell (for live websockets logs), execute this cell. If you prefer utilizing Colab's native sidebar Terminal with `tmux`, skip this cell and refer to Step 5.

In [ ]:
train_cmd = """
PYTHONPATH=/content/drive/MyDrive/sutta-tts-model-training/scripts:$PYTHONPATH \
python3 -m piper.train fit \
  --data.voice_name "en_gb-suttaplayer-medium" \
  --data.csv_path "/content/drive/MyDrive/sutta-tts-model-training/corpus-preperation/metadata-phonemes.csv" \
  --data.phoneme_type text \
  --data.phonemes_path "/content/drive/MyDrive/sutta-tts-model-training/config/en[gb]_pi[si]-suttaplayer-phoneme-map.json" \
  --data.audio_dir "/content/drive/MyDrive/piper_training/wavs" \
  --model.sample_rate 22050 \
  --data.espeak_voice "en-gb" \
  --data.cache_dir "/content/piper_cache" \
  --data.config_path "/content/drive/MyDrive/piper_training/en_gb-suttaplayer-medium.json" \
  --data.batch_size 8 \
  --trainer.callbacks.class_path "train_sutta_voice.SuttaVoiceUatCallback" \
  --ckpt_path "/content/drive/MyDrive/piper_training/last-9288-57498.ckpt" \
  --trainer.accelerator gpu --trainer.devices 1 --trainer.precision 16-mixed \
  --model.mel_fmin 0 \
  --model.mel_fmax 8000
"""

# Ingest, parse, and enforce strict pre-flight gate bounds!
validator = PreFlightValidator(train_cmd)
validator.run_audit()

# This will launch the training process if and only if validator.run_audit() is 100% successful!
# validator.train()

### Step 5: Active UAT Keep-Alive & Playback Dashboard
Run this cell in your notebook to keep your Colab session active. It synchronizes your `uat_metrics.csv` straight to your Google Sheet (`SuttaPlayer_UAT_Convergence`) every 30 seconds, and dynamically renders HTML5 audio playback widgets for your local NVMe previews—generating **exactly 0 bytes of Google Drive Trash**!

💡 **SUPPRESS BROWSER TIMEOUTS (AUTO-CLICKER JS):**
1. Press `F12` (or right-click and select Inspect) inside Brave/Chrome to open Developer Tools.
2. Click on the **Console** tab.
3. Paste the following JavaScript and press Enter:
```js
function KeepAlive() {
  let connectBtn = document.querySelector("#connect") || document.querySelector("colab-connect-button");
  if (connectBtn) {
    console.log("Simulating click on Connect Button...");
    connectBtn.click();
  }
}
setInterval(KeepAlive, 60000); // Triggers every 60 seconds
```

In [ ]:
# Launches the Keep-Alive Sheet Sync & Local HTML5 Audio Playback Dashboard
!python3 /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-dashboard.py

### Step 6: Native Colab Terminal & TMUX splitting Guide
Google Colab now natively supports persistent background terminals. You can use the Terminal tab at the bottom-left of your sidebar to manage everything securely under `tmux` so that browser-reloads or dropped tabs never interrupt your session!

#### Terminal Pane Setup Sequence:

1. Open the **Terminal Tab** in the lower-left sidebar.
2. Spin up a new background terminal session:
   ```bash
   tmux new -s suttaplayer
   ```
3. **Split your pane horizontally** (or vertically) by pressing `Ctrl+B` then `"` (double quote).
4. In the **Foreground Pane (Pane 1)**, launch your validated trainer:
   ```bash
   # Paste your training command from Step 4 here!
   PYTHONPATH=/content/drive/MyDrive/sutta-tts-model-training/scripts:$PYTHONPATH python3 -m piper.train fit ...
   ```
5. Press `Ctrl+B` then `O` (or the arrow keys) to jump to the **Background Pane (Pane 2)** and launch your trash-free sync pruner:
   ```bash
   deno run --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-pruner.ts
   ```
6. If you want to temporarily detach from the terminal and let it run completely independently, press `Ctrl+B` then `D`. Re-attach anytime with `tmux a -t suttaplayer`!